# Gesture Recognition System

In [50]:
import cv2 as cv
import numpy as np
import math

In [53]:
# Gesture recognition using convex hull and contour detection
cap = cv.VideoCapture(0)

def b(n):
    pass

if not cap.isOpened():
    raise RuntimeError("Could not open video device")
# crate Trackbar
cv.namedWindow('Threshold Parameter')
cv.resizeWindow('Threshold Parameter', 640, 240)
cv.createTrackbar('Threshold', 'Threshold Parameter', 150, 255, b)

while True:
    ret, frame = cap.read()
    if not ret:
        raise RuntimeError("Could not read from video device")
    #    
    frame = cv.flip(frame, 1)

    # Define Region of Interest (ROI)
    roi = frame[100:300, 100:300]
    cv.rectangle(frame, (100, 100), (300, 300), (0, 255, 0), 2)
    # convert to gray and apply blur
    roi_gray = cv.cvtColor(roi, cv.COLOR_BGR2GRAY)
    blur = cv.GaussianBlur(roi_gray, (9, 9), 0)
    
    value = cv.getTrackbarPos('Threshold', 'Threshold Parameter')
    _, thresh = cv.threshold(blur, value, 255, cv.THRESH_BINARY_INV) # Uses INV to capture hand as white

    # Find Contours
    version = cv.__version__.split('.')[0]
    if version == '3':
        _, contours, _ = cv.findContours(thresh, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
    else:
        contours, _ = cv.findContours(thresh, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)

    if len(contours) > 0:
        try:
            # Get largest contour (the hand)
            max_contour = max(contours, key=lambda x: cv.contourArea(x))
            
            # returnPoints=False is required to calculate convexity defects
            hull_indices = cv.convexHull(max_contour, returnPoints=False)
            hull_points = cv.convexHull(max_contour, returnPoints=True)
            
            cv.drawContours(roi, [hull_points], -1, (0, 255, 0), 2)
            cv.drawContours(roi, [max_contour], -1, (255, 0, 0), 2)
            
            defect_count = 0

            # Calculate defects if hull has enough points
            if len(hull_indices) > 3:
                defects = cv.convexityDefects(max_contour, hull_indices)
                
                if defects is not None:
                    for i in range(defects.shape[0]):
                        s, e, f, d = defects[i, 0]
                        start = tuple(max_contour[s][0])
                        end = tuple(max_contour[e][0])
                        far = tuple(max_contour[f][0])

                        # Calculate triangle sides
                        a = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
                        b = math.sqrt((far[0] - start[0])**2 + (far[1] - start[1])**2)
                        c = math.sqrt((end[0] - far[0])**2 + (end[1] - far[1])**2)

                        # Cosine rule & conversion to degrees
                        cosine_val = (b**2 + c**2 - a**2) / (2 * b * c)
                        cosine_val = max(-1.0, min(1.0, cosine_val)) # Clip boundaries to avoid math domain errors
                        angle = math.acos(cosine_val) * 57.2958 # Convert to degrees

                        # If angle between fingers is less than 90 degrees, it is a finger space
                        if angle <= 90 and d > 1000: # d > 1000 filters minor noise defects
                            defect_count += 1
                            cv.circle(roi, far, 5, [0, 0, 255], -1)
                        
                        cv.line(roi, start, end, [0, 255, 0], 2)

            # Display the number based on the spaces (defects) between fingers
            # 0 defects = 1 finger, 1 defect = 2 fingers, etc.
            display_text = str(defect_count + 1) if defect_count < 5 else "Reposition"
            cv.putText(frame, display_text, (50, 50), cv.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
                
        except Exception as e:
            # Prints errors to the terminal instead of hiding them
            print(f"Error tracking hand: {e}")
            
    cv.imshow('Original', frame)
    cv.imshow('Threshold', thresh)
    cv.imshow('ROI', roi)
    
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()
